In [ ]:
import os
import re
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [ ]:
def read_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def extract_operators(text):
    operators = re.findall(r'\[(AND|OR|NOT)\]', text)
    return operators

def count_operators(operators):
    return {
        'AND': operators.count('AND'),
        'OR': operators.count('OR'),
        'NOT': operators.count('NOT')
    }

def compare_operators(label_text, model_text):
    label_operators = extract_operators(label_text)
    model_operators = extract_operators(model_text)
    label_counts = count_operators(label_operators)
    model_counts = count_operators(model_operators)

    relaxed_comparison = {
        'AND': label_counts['AND'] == model_counts['AND'],
        'OR': label_counts['OR'] == model_counts['OR'],
        'NOT': label_counts['NOT'] == model_counts['NOT']
    }

    def exact_position_comparison(label_text, model_text, operator):
        pattern = re.compile(rf'(\w+)\s+\[{operator}\]')
        label_positions = [(match.group(1), match.start()) for match in pattern.finditer(label_text)]
        model_positions = [(match.group(1), match.start()) for match in pattern.finditer(model_text)]


        return label_positions == model_positions

    exact_comparison = {
        'AND': exact_position_comparison(label_text, model_text, 'AND'),
        'OR': exact_position_comparison(label_text, model_text, 'OR'),
        'NOT': exact_position_comparison(label_text, model_text, 'NOT')
    }

    return relaxed_comparison, exact_comparison, label_counts, model_counts

def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

def process_files(label_folder, model_folder):
    relaxed_comparisons = []
    exact_comparisons = []
    label_operator_counts = []
    model_operator_counts = []

    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)

            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)

                if os.path.exists(label_file_path):
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)

                    relaxed_comparison, exact_comparison, label_counts, model_counts = compare_operators(label_text, model_text)
                    print(label_counts)
                    print(model_counts)

                    relaxed_comparisons.append(relaxed_comparison)
                    exact_comparisons.append(exact_comparison)
                    label_operator_counts.append(label_counts)
                    model_operator_counts.append(model_counts)

    return relaxed_comparisons, exact_comparisons, label_operator_counts, model_operator_counts

In [ ]:
label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'

relaxed_comparisons, exact_comparisons, label_operator_counts, model_operator_counts = process_files(label_folder, model_folder)

In [ ]:
total_and = sum(count['AND'] for count in label_operator_counts)
total_or = sum(count['OR'] for count in label_operator_counts)
total_not = sum(count['NOT'] for count in label_operator_counts)

print("Total AND:", total_and)
print("Total OR:", total_or)
print("Total NOT:", total_not)

In [ ]:
total_and = sum(count['AND'] for count in model_operator_counts)
total_or = sum(count['OR'] for count in model_operator_counts)
total_not = sum(count['NOT'] for count in model_operator_counts)

print("Total AND:", total_and)
print("Total OR:", total_or)
print("Total NOT:", total_not)

### Zähle gleich viele Operatoren im gleichen Text

In [ ]:
import os
import re
from collections import defaultdict

def read_file(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        return file.read()

def extract_operators(text):
    operators = re.findall(r'\[(AND|OR|NOT)\]', text)
    return operators

def count_operators(operators):
    return {
        'AND': operators.count('AND'),
        'OR': operators.count('OR'),
        'NOT': operators.count('NOT')
    }

def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

def calculate_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

    return precision, recall, f1, accuracy

def process_files(label_folder, model_folder):
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)
    label_counts_total = defaultdict(int)
    model_counts_total = defaultdict(int)
    matching_files = defaultdict(list)

    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)
            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)
                if os.path.exists(label_file_path):
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)

                    label_operators = extract_operators(label_text)
                    model_operators = extract_operators(model_text)

                    label_counts = count_operators(label_operators)
                    model_counts = count_operators(model_operators)

                    for op in ['AND', 'OR', 'NOT']:
                        label_counts_total[op] += label_counts.get(op, 0)
                        model_counts_total[op] += model_counts.get(op, 0)
                        if label_counts.get(op, 0) == model_counts.get(op, 0) and label_counts.get(op, 0) != 0:
                            tp[op] += 1
                            matching_files[op].append(nct_number)  # Speichere die NCT-Nummer
                        else:
                            if label_counts.get(op, 0) > model_counts.get(op, 0):
                                fn[op] += 1
                            else:
                                fp[op] += 1

    for op in ['AND', 'OR', 'NOT']:
        precision, recall, f1, accuracy = calculate_metrics(tp[op], fp[op], fn[op])
        print(10*"--")
        print(f"{op}:")
        print(f"  Precision: {precision:.3f}")
        print(f"  Recall: {recall:.3f}")
        print(f"  F1-score: {f1:.3f}")
        print(f"  Accuracy: {accuracy:.3f}")
        print(10*"--")
        print(f"  Total in Label: {label_counts_total[op]}")
        print(f"  Total in Model: {model_counts_total[op]}")
        print(f"  Correctly Identified: {tp[op]}  (gleiche Anzahl pro file)")
        #print(f"  False Positives: {fp[op]}")
        #print(f"  False Negatives: {fn[op]}")
        #print(f"  Matching Files: {', '.join(matching_files[op])}")
        print(10*"--")

In [ ]:
label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'
process_files(label_folder, model_folder)

In [ ]:
label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-8B-Instruct_4_shot/output'
process_files(label_folder, model_folder)